# Project 1: VaR and ES -- Estimation, Backtesting, and Model Comparison

**Names of all group members:**
- Firstname Lastname (email@example.com)
- Firstname Lastname (email@example.com)

---

## 0) Setup

Creates necessary folders, sets parameters, and fixes the random seed. Keep this seed fixed; the whole run must be reproducible from it.

In [ ]:
# If a package import fails, install it in your environment, e.g.:
# %pip install arch statsmodels

import os
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib.pyplot as plt
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_FILE = PROJECT_DIR / "QRM_2026_Project_prices.csv"   # provided with the project; do not re-download
OUT_DIR = PROJECT_DIR / "output"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Parameters
TICKERS = ["AAPL", "META", "JPM"]   # every question is run separately on each stock
WINDOW = 504                        # estimation window of every model, in trading days (Questions 2-5)
EVAL_START = 504                    # Questions 3-5: forecast days are all days after the first 504
REFIT = 21                          # Questions 4-5: re-estimate every this many trading days
ALPHAS = [0.975, 0.99]              # confidence levels used throughout Questions 2-5
DEEP_ALPHA = 0.999                  # Question 6: the extreme level EVT is asked for
M_SIM = 10_000                      # Questions 3(c), 4(d): Acerbi-Szekely simulated samples
LB_LAGS = 10                        # Ljung-Box H in Questions 1 and 4
NW_LAGS = 10                        # Newey-West truncation lag of the Diebold-Mariano test
SEED = 0                            # fix this, state it in the report, do not change it later

rng = np.random.default_rng(SEED)
print("Project dir:", PROJECT_DIR)


## 1) Data

The prices were downloaded once with `yfinance` (adjusted closes, 2 January 2013 to 30 June 2026,
downloaded on 28 October 2026) and are provided as `QRM_2026_Project_prices.csv`. Read this file;
do not download the data again, since Yahoo revises adjusted prices back in time.

In [ ]:
price_df = pd.read_csv(DATA_FILE, index_col=0, parse_dates=True)[TICKERS]
print(price_df.shape, price_df.index[0].date(), "to", price_df.index[-1].date())   # (3393, 3)

## 2) Log-returns and losses

Construct $R_t=\log(P_t/P_{t-1})$ for each ticker. The log-loss used everywhere in this project is
$L_t=-R_t$. Every question is run separately on each of the three columns; there is no portfolio.

In [ ]:
returns = np.log(price_df / price_df.shift(1)).dropna()
losses = -returns

print(returns.shape, returns.index[0].date(), "to", returns.index[-1].date())   # (3392, 3)
returns.head()

## Question 1 -- Stylized facts

For each stock: (a) plot the return series; (b) ACF of returns and of squared returns, and the
Ljung-Box test with $H$ = `LB_LAGS` on both (`statsmodels.stats.diagnostic.acorr_ljungbox`);
(c) QQ-plot and Jarque-Bera.

In [ ]:
# Question 1


## Question 2 -- Single-window VaR and ES estimates

For each stock, use the first `WINDOW` losses as the single estimation window for (a)-(d).

In [ ]:
# Question 2
window_loss = losses.iloc[:WINDOW]          # first 504 losses, one column per stock


# ============ 1. Functions: each returns (VaR, ES) ============

def hs_var_es(x, alpha):
    """(a) Historical simulation."""
    L = np.sort(x)[::-1]                    # sort losses from largest to smallest
    W = len(L)                              # 504
    k = int(np.floor(W * (1 - alpha)))      # number of losses strictly beyond the VaR
    var = L[k]                              # the (k+1)-th largest loss (Python starts counting at 0)
    es = (L[:k].sum() + (W * (1 - alpha) - k) * L[k]) / (W * (1 - alpha))
    return var, es


def gauss_var_es(mu, sigma, alpha):
    """(b) VaR and ES of a normal distribution N(mu, sigma^2)."""
    z = stats.norm.ppf(alpha)               # alpha-quantile of N(0,1)
    var = mu + sigma * z
    es = mu + sigma * stats.norm.pdf(z) / (1 - alpha)
    return var, es


def t_fit(x):
    """(c) Fit a Student-t (nu, mu, s) by maximum likelihood.
    t.fit can get stuck at a local optimum, so try 4 starting values of nu
    and keep the fit with the highest log-likelihood."""
    best_fit, best_loglik = None, -np.inf
    for nu_start in [3, 5, 10, 30]:
        fit = stats.t.fit(x, nu_start)                  # (nu, mu, s)
        loglik = stats.t.logpdf(x, *fit).sum()          # how well this fit explains the data
        if loglik > best_loglik:
            best_fit, best_loglik = fit, loglik
    return best_fit


def t_var_es(nu, mu, s, alpha):
    """(c) VaR and ES of a Student-t with location mu and scale s."""
    q = stats.t.ppf(alpha, nu)              # alpha-quantile of the standard t
    var = mu + s * q
    es = mu + s * stats.t.pdf(q, nu) / (1 - alpha) * (nu + q**2) / (nu - 1)
    return var, es


# ============ 2. Apply the three methods to each stock ============

rows = []                                   # one row per (stock, alpha, method)
fits = {}                                   # fitted parameters, kept for the plot

for tic in TICKERS:
    x = window_loss[tic].values

    mu_g, sd_g = x.mean(), x.std(ddof=1)    # Gaussian: sample mean and std
    nu, mu_t, s_t = t_fit(x)                # Student-t: maximum likelihood
    fits[tic] = (mu_g, sd_g, nu, mu_t, s_t)
    print(f"{tic}: Gaussian mu={mu_g:.5f}, sd={sd_g:.5f} | "
          f"Student-t nu={nu:.2f}, mu={mu_t:.5f}, scale={s_t:.5f}")

    for a in ALPHAS:
        var, es = hs_var_es(x, a)
        rows.append([tic, a, "HS", var, es])
        var, es = gauss_var_es(mu_g, sd_g, a)
        rows.append([tic, a, "Gaussian", var, es])
        var, es = t_var_es(nu, mu_t, s_t, a)
        rows.append([tic, a, "Student-t", var, es])

results = pd.DataFrame(rows, columns=["stock", "alpha", "method", "VaR", "ES"])

# Print one small table per level: rows = method, columns = stock, in %
for a in ALPHAS:
    sub = results[results.alpha == a]
    print(f"\nalpha = {a}, VaR (%):")
    print((100 * sub.pivot(index="method", columns="stock", values="VaR")).round(2))
    print(f"alpha = {a}, ES (%):")
    print((100 * sub.pivot(index="method", columns="stock", values="ES")).round(2))

results.to_csv(OUT_DIR / "q2_var_es.csv", index=False)


# ============ 3. (d) Plot: histogram + two densities + VaR lines ============

colors = {"HS": "black", "Gaussian": "tab:blue", "Student-t": "tab:red"}
styles = {0.975: "--", 0.99: ":"}           # dashed = 97.5%, dotted = 99%

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, tic in zip(axes, TICKERS):
    x = window_loss[tic].values
    mu_g, sd_g, nu, mu_t, s_t = fits[tic]
    grid = np.linspace(x.min(), x.max(), 1000)          # x-values where we draw the pdfs

    ax.hist(x, bins=60, density=True, color="lightgray", label="Empirical")
    ax.plot(grid, stats.norm.pdf(grid, mu_g, sd_g), color="tab:blue", label="Gaussian pdf")
    ax.plot(grid, stats.t.pdf(grid, nu, mu_t, s_t), color="tab:red", label="Student-t pdf")

    # one vertical line per VaR estimate of this stock
    r = results[results.stock == tic]
    for method, a, var in zip(r.method, r.alpha, r.VaR):
        ax.axvline(var, color=colors[method], ls=styles[a], label=f"VaR {method} {a}")

    ax.set_title(f"{tic} (nu = {nu:.2f})")
    ax.set_xlabel("log-loss")

axes[0].set_ylabel("density")
axes[0].legend(fontsize=7)                  # same legend for all three panels, show it once
plt.tight_layout()
plt.savefig(OUT_DIR / "q2_densities.png", dpi=200)
plt.show()

## Question 3 -- Backtesting

Forecast every day of the evaluation period `losses.iloc[EVAL_START:]`, each model estimated on the
`WINDOW` days before the forecast day. Historical simulation, Gaussian and Student-$t$ are recomputed
every day. Write the Kupiec, Christoffersen, Acerbi-Szekely, Fissler-Ziegel, pinball-loss and
Diebold-Mariano functions yourself, from the course slides (Kupiec, Christoffersen) and Appendix A of
the project statement (the others); they are
reused unchanged in Questions 4 and 5.

The combined VaR backtest table collects the VaR backtests of Questions 3-5: append one row per
stock, method and level to `backtest_rows` as you go. The ES backtest table ($\alpha=0.975$) collects
the Acerbi-Szekely and Fissler-Ziegel results of Questions 3 and 4 in `es_rows`.

In [ ]:
# Question 3
eval_loss = losses.iloc[EVAL_START:]        # one column per stock, 2888 forecast days
backtest_rows = []                          # dicts: stock, method, alpha, x, expected, hit, p_uc, p_ind, p_cc, pinball_bp
es_rows = []                                # dicts: stock, method, Z, p_Z, FZ


## Question 4 -- AR-GARCH conditional VaR and ES

(a) Mean model, full sample: fit AR($p$), $p=0,\dots,3$, with constant variance on the same
observations, e.g. `ARX(y, lags=p, hold_back=3)` from `arch.univariate`; pick $p$ by BIC (also report
AIC); Ljung-Box on the residuals (`model_df=p`) and on their squares.
(b) Fit `arch_model(y, mean="AR", lags=p, vol="GARCH", p=1, q=1, dist=..., hold_back=3)` on the full
sample with `dist="normal"` and `dist="t"` (use `mean="Constant"` if $p=0$); Ljung-Box on the
standardized residuals of the Gaussian fit (`model_df=p`) and their squares (`model_df=2`).
(c) Re-estimate both inside the rolling scheme every `REFIT` days with the same $p$. Between refits,
keep the parameters fixed but advance $\hat\mu_t$ and $\hat\sigma_t$ daily with the realized
returns (for example with `arch_model(...).fix(params)` on the data up to the forecast day). Fit on
returns in percent (`100 * R`) to avoid `arch` convergence warnings, and convert the forecasts back.
(d) Backtest both and add them to the two tables.

In [ ]:
# Question 4
from arch import arch_model


## Question 5 -- Quantile regression

Same evaluation period, window and refit days as Question 4. One model per stock and level, VaR only;
the two features are built by `make_features` below, both using information up to $t-1$; the target
is the loss $L_t$. Rows with an undefined feature (the first 20 days) are dropped from the training
window. Linear quantile regression, e.g.
`sm.QuantReg(y, sm.add_constant(X)).fit(q=alpha, max_iter=5000)`.

In [ ]:
# Question 5
import statsmodels.api as sm


def make_features(ret: pd.Series) -> pd.DataFrame:
    """Features x_{t-1} for the forecast of day t: row t uses information up to t-1 only."""
    feat = pd.DataFrame(index=ret.index)
    feat["absR_lag1"] = ret.shift(1).abs()
    feat["sigma20"] = ret.shift(1).rolling(20).std(ddof=1)
    return feat


# After (a) and (b): the combined VaR backtest table, one table per level
# backtest_table = pd.DataFrame(backtest_rows).set_index(["alpha", "stock", "method"]).sort_index()


## Question 6 -- Extreme value theory

One static fit on each stock's full-sample log-loss series, with threshold $u$ the empirical 95%
quantile. `scipy.stats.genpareto.fit(excesses, floc=0)` may be used for the GPD maximum-likelihood
step.

In [ ]:
# Question 6
from scipy.stats import genpareto
full_loss = losses                          # one column per stock
